# Phase 39 — Stacking + Label Smoothing

## ΜΕΡΟΣ Α: Stacking Ensemble
Αντί για απλό weighted average, εκπαιδεύουμε meta-classifier
που μαθαίνει πώς να συνδυάσει τα probs των μοντέλων.

```
BERT-base probs  (10d hazard, 22d product) ]
Multi-task probs (10d hazard, 22d product) ] → LogReg → final prediction
```

## ΜΕΡΟΣ Β: Label Smoothing + Focal Loss
Αντί για hard labels (0/1), χρησιμοποιούμε soft labels:
```
Hard:   [0, 0, 1, 0, 0] (one-hot)
Smooth: [0.01, 0.01, 0.94, 0.01, 0.01] (smoothed)
```
Μειώνει overconfidence και βελτιώνει generalization.

**Best so far:** 0.81882

# ═══════════════════════════════════
# ΜΕΡΟΣ Α — STACKING
# ═══════════════════════════════════

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_predict
import warnings
warnings.filterwarnings('ignore')

In [3]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

train_full = pd.concat([train, valid], ignore_index=True)

le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

# Φόρτωση validation probs (train-only models)
# από phase15 (BERT-base train-only)
bert_valid_hazard_probs  = np.load('npy/bertbase_focal_trainonly_valid_hazard_probs.npy')
bert_valid_product_probs = np.load('npy/bertbase_focal_trainonly_valid_product_probs.npy')


# Test probs (train+valid models)
bert_test_hazard_probs    = np.load('npy/bertbase_focal_test_hazard_probs2.npy')
bert_test_product_probs   = np.load('npy/bertbase_focal_test_product_probs2.npy')
multi_test_hazard_probs   = np.load('npy/multitask_test_hazard_probs.npy')
multi_test_product_probs  = np.load('npy/multitask_test_product_probs.npy')

print(f'BERT valid hazard probs:  {bert_valid_hazard_probs.shape}')
print(f'BERT test hazard probs:   {bert_test_hazard_probs.shape}')

BERT valid hazard probs:  (565, 10)
BERT test hazard probs:   (997, 10)


In [4]:
def official_st1_score(y_true_h, y_pred_h, y_true_p, y_pred_p, verbose=True):
    f1_h = f1_score(y_true_h, y_pred_h, average='macro', zero_division=0)
    mask = (np.array(y_true_h) == np.array(y_pred_h))
    f1_p = f1_score(
        np.array(y_true_p)[mask], np.array(y_pred_p)[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_h + f1_p) / 2
    if verbose:
        print(f'  macro-F1 Hazard:                    {f1_h:.4f}')
        print(f'  Σωστά hazard:                       {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'  macro-F1 Product (given correct h): {f1_p:.4f}')
        print(f'  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'  OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

In [5]:
# === STACKING — HAZARD ===
# Meta-features: BERT probs (validation)
X_meta_valid_hazard = bert_valid_hazard_probs  # μόνο BERT για τώρα
y_valid_hazard      = le_hazard.transform(valid['hazard-category'])

print('=== STACKING — HAZARD ===')
print(f'Meta-features shape: {X_meta_valid_hazard.shape}\n')

# Δοκιμή διαφορετικών meta-classifiers
meta_classifiers = [
    ('LogReg (C=1)',    LogisticRegression(C=1,   max_iter=1000, random_state=42)),
    ('LogReg (C=0.1)', LogisticRegression(C=0.1, max_iter=1000, random_state=42)),
    ('LogReg (C=10)',  LogisticRegression(C=10,  max_iter=1000, random_state=42)),
]

best_meta_h = None
best_f1_h   = 0

for name, clf in meta_classifiers:
    # Cross-val για να μην overfit
    preds = cross_val_predict(clf, X_meta_valid_hazard, y_valid_hazard, cv=5)
    f1    = f1_score(y_valid_hazard, preds, average='macro', zero_division=0)
    print(f'{name:20s}: CV F1 Hazard = {f1:.4f}')
    if f1 > best_f1_h:
        best_f1_h   = f1
        best_meta_h = clf

# Εκπαίδευση στο σύνολο validation
best_meta_h.fit(X_meta_valid_hazard, y_valid_hazard)
print(f'\nΒέλτιστο: CV F1 Hazard = {best_f1_h:.4f}')

=== STACKING — HAZARD ===
Meta-features shape: (565, 10)

LogReg (C=1)        : CV F1 Hazard = 0.3534
LogReg (C=0.1)      : CV F1 Hazard = 0.1431
LogReg (C=10)       : CV F1 Hazard = 0.5381

Βέλτιστο: CV F1 Hazard = 0.5381


In [6]:
# === STACKING — PRODUCT ===
X_meta_valid_product = bert_valid_product_probs
y_valid_product      = le_product.transform(valid['product-category'])

print('=== STACKING — PRODUCT ===')
print(f'Meta-features shape: {X_meta_valid_product.shape}\n')

best_meta_p = None
best_f1_p   = 0

for name, clf in [
    ('LogReg (C=1)',    LogisticRegression(C=1,   max_iter=1000, random_state=42)),
    ('LogReg (C=0.1)', LogisticRegression(C=0.1, max_iter=1000, random_state=42)),
    ('LogReg (C=10)',  LogisticRegression(C=10,  max_iter=1000, random_state=42)),
]:
    preds = cross_val_predict(clf, X_meta_valid_product, y_valid_product, cv=5)
    f1    = f1_score(y_valid_product, preds, average='macro', zero_division=0)
    print(f'{name:20s}: CV F1 Product = {f1:.4f}')
    if f1 > best_f1_p:
        best_f1_p   = f1
        best_meta_p = clf

best_meta_p.fit(X_meta_valid_product, y_valid_product)
print(f'\nΒέλτιστο: CV F1 Product = {best_f1_p:.4f}')

=== STACKING — PRODUCT ===
Meta-features shape: (565, 22)

LogReg (C=1)        : CV F1 Product = 0.3391
LogReg (C=0.1)      : CV F1 Product = 0.0228
LogReg (C=10)       : CV F1 Product = 0.5255

Βέλτιστο: CV F1 Product = 0.5255


In [7]:
# Test predictions με stacking
# Meta-features για test: BERT test probs
stacking_test_hazard  = le_hazard.inverse_transform(
    best_meta_h.predict(bert_test_hazard_probs)
)
stacking_test_product = le_product.inverse_transform(
    best_meta_p.predict(bert_test_product_probs)
)

pd.DataFrame({
    'id': test['id'],
    'hazard-category': stacking_test_hazard,
    'product-category': stacking_test_product
}).to_csv('submission_stacking.csv', index=False)
print('Αποθηκεύτηκε: submission_stacking.csv')

Αποθηκεύτηκε: submission_stacking.csv


# ═══════════════════════════════════════
# ΜΕΡΟΣ Β — LABEL SMOOTHING
# ═══════════════════════════════════════

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Επαναφόρτωση δεδομένων για Μέρος Β
train = pd.read_csv('/train.csv')
valid = pd.read_csv('/valid.csv')
test  = pd.read_csv('/test.csv')

train_full = pd.concat([train, valid], ignore_index=True)

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_full = make_text(train_full)
texts_test = make_text(test)

le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

y_full_hazard  = le_hazard.transform(train_full['hazard-category'])
y_full_product = le_product.transform(train_full['product-category'])

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)

dummy_labels = np.zeros(len(texts_test), dtype=int)

print(f'Train+Valid: {len(train_full)} | Test: {len(texts_test)}')
print(f'Hazard: {n_hazard} | Product: {n_product}')

In [ ]:
class FocalLossWithLabelSmoothing(nn.Module):
    """
    Συνδυασμός Focal Loss + Label Smoothing.

    Label Smoothing: αντί για hard labels (0/1) χρησιμοποιεί:
    - Correct class:   1 - smoothing + smoothing/n_classes
    - Wrong classes:   smoothing/n_classes

    Παράδειγμα (smoothing=0.1, n=10):
    Hard:   [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
    Smooth: [0.01, 0.01, 0.91, 0.01, 0.01, ...]
    """
    def __init__(self, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.gamma     = gamma
        self.smoothing = smoothing

    def forward(self, logits, targets):
        n_classes = logits.size(1)

        # Smooth labels
        with torch.no_grad():
            smooth_targets = torch.full_like(logits, self.smoothing / n_classes)
            smooth_targets.scatter_(1, targets.unsqueeze(1),
                                    1.0 - self.smoothing + self.smoothing / n_classes)

        # Log probabilities
        log_probs = F.log_softmax(logits, dim=1)
        probs     = torch.exp(log_probs)

        # Cross entropy με smooth labels
        ce = -(smooth_targets * log_probs).sum(dim=1)

        # Focal weight — βάσει original (non-smoothed) targets
        pt     = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal  = (1 - pt) ** self.gamma * ce

        return focal.mean()


print('FocalLossWithLabelSmoothing defined!')
print('Παράδειγμα (smoothing=0.1, n=10):')
print('  Hard label:   [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]')
print('  Smooth label: [0.01, 0.01, 0.91, 0.01, ...]')

In [ ]:
MODEL_NAME   = 'bert-base-uncased'
BATCH_SIZE   = 32
MAX_LENGTH   = 128
LR           = 2e-5
N_EPOCHS     = 20
WARMUP_RATIO = 0.1
SMOOTHING    = 0.1  # Label smoothing factor

criterion_smooth = FocalLossWithLabelSmoothing(gamma=2.0, smoothing=SMOOTHING)
tokenizer        = AutoTokenizer.from_pretrained(MODEL_NAME)
dummy_labels     = np.zeros(len(texts_test), dtype=int)

print(f'Model: {MODEL_NAME}')
print(f'Label smoothing: {SMOOTHING}')
print(f'Focal gamma: 2.0')

In [ ]:
class FoodHazardDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts; self.labels = labels
        self.tokenizer = tokenizer; self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), max_length=self.max_length,
                              padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}


def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        labels = batch.pop('label').to(device)
        batch  = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        loss = criterion(model(**batch).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def get_probabilities(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            probs = torch.softmax(model(**batch).logits, dim=1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)


full_loader_hazard  = DataLoader(FoodHazardDataset(texts_full, y_full_hazard,  tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
full_loader_product = DataLoader(FoodHazardDataset(texts_full, y_full_product, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
test_loader_hazard  = DataLoader(FoodHazardDataset(texts_test, dummy_labels,   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
test_loader_product = DataLoader(FoodHazardDataset(texts_test, dummy_labels,   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(full_loader_hazard)}')

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ BERT-base + Focal + Label Smoothing — HAZARD ===')

bert_hazard = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_hazard).to(device)
opt_h       = AdamW(bert_hazard.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(full_loader_hazard) * N_EPOCHS
sch_h       = get_linear_schedule_with_warmup(opt_h, int(total_steps*WARMUP_RATIO), total_steps)

for epoch in range(N_EPOCHS):
    loss = train_epoch(bert_hazard, full_loader_hazard, opt_h, sch_h, criterion_smooth)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {loss:.4f}')

print('Hazard ολοκληρώθηκε')

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ BERT-base + Focal + Label Smoothing — PRODUCT ===')

bert_product = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_product).to(device)
opt_p        = AdamW(bert_product.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(full_loader_product) * N_EPOCHS
sch_p        = get_linear_schedule_with_warmup(opt_p, int(total_steps*WARMUP_RATIO), total_steps)

for epoch in range(N_EPOCHS):
    loss = train_epoch(bert_product, full_loader_product, opt_p, sch_p, criterion_smooth)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {loss:.4f}')

print('Product ολοκληρώθηκε')

In [ ]:
# Test predictions
smooth_hazard_probs  = get_probabilities(bert_hazard,  test_loader_hazard)
smooth_product_probs = get_probabilities(bert_product, test_loader_product)

np.save('bertbase_smooth_test_hazard_probs.npy',  smooth_hazard_probs)
np.save('bertbase_smooth_test_product_probs.npy', smooth_product_probs)

test_hazard  = le_hazard.inverse_transform(smooth_hazard_probs.argmax(axis=1))
test_product = le_product.inverse_transform(smooth_product_probs.argmax(axis=1))

pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_hazard,
    'product-category': test_product
}).to_csv('submission_label_smoothing.csv', index=False)

print('Αποθηκεύτηκε: submission_label_smoothing.csv')
print('Αποθηκεύτηκαν .npy για ensemble')
print('\n=== ΣΥΓΚΡΙΣΗ ===')
print('BERT-base Focal (no smoothing): 0.80399')
print('Best ensemble:                  0.81882')